In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, lit, round, current_timestamp

In [0]:
src_Table='project_mobility.bronze.weather_raw'
tgt_Table='project_mobility.silver.weather_clean'


In [0]:
df=spark.read.table(src_Table)

df_selected = df.select(
      col("DATE").alias("date"),
      col("TMAX").alias("temp_max"),
      col("TMIN").alias("temp_min"),
      col("TAVG").cast("int").alias("tavg_raw"),
      col("PRCP").alias("precipitation"),
      col("SNOW").alias("snowfall"),
      col("SNWD").alias("snow_depth"),
      col("AWND").alias("avg_wind_speed"),
      when(col("WT01") == 1, True).otherwise(False).alias("has_fog"),
      when(col("WT03") == 1, True).otherwise(False).alias("has_thunder"),
      when(col("WT04") == 1, True).otherwise(False).alias("has_ice_sleet")
  )

df_nulls = df_selected.fillna({
      "precipitation": 0.0,
      "snowfall": 0.0,
      "snow_depth": 0.0
  })

df_derived = df_nulls.withColumns({
      "temp_avg": when(col("tavg_raw").isNotNull(), col("tavg_raw")
      ).otherwise(round((col("temp_max") + col("temp_min")) / 2, 1)),
      "is_rainy":  col("precipitation") > 0,
      "is_snowy":  col("snowfall") > 0,
      "weather_condition": when(col("snowfall") > 2,"Heavy Snow")
                          .when(col("snowfall") > 0,"Snow")
                          .when(col("precipitation") > 0.5,"Heavy Rain")
                          .when(col("precipitation") > 0,"Rain")
                          .otherwise("Clear"),
      "temp_category": when(col("temp_avg") < 32,"Cold")
                      .when(col("temp_avg") < 50,"Cool")
                      .when(col("temp_avg") < 70,"Mild")
                      .otherwise("Warm"),
      "ingestion_timestamp": current_timestamp()
  })

df_final = df_derived.drop("tavg_raw")

df_final.write.mode("overwrite").saveAsTable(tgt_Table)